# 07 — Análisis de ablaciones P1–P4

**Protocolo:** análisis exclusivamente de archivos existentes. No se entrenan modelos ni se modifican los CSV originales. Las columnas se inspeccionan antes de utilizarlas. Las métricas no se reinterpretan automáticamente sin conocer su definición.

In [5]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = Path(r"C:\Users\hugoc\Local\PUCV\Doctorado\Cursos\Semestre-2-2026\Industria-Inteligente\archive\resultados_hai_arboles")
assert ROOT.is_dir(), f"No existe carpeta: {ROOT}"
DEST = ROOT / 'analisis'
DEST.mkdir(exist_ok=True)
FILES = sorted(p for p in ROOT.rglob('*.csv') if DEST not in p.parents)
print('CSV encontrados:', len(FILES))
def load_csv(p):
    for enc in ('utf-8-sig','latin1'):
        try:
            df=pd.read_csv(p,encoding=enc,sep=None,engine='python')
            return df
        except UnicodeDecodeError: continue
    raise ValueError(f'No se pudo leer {p}')
def inventory():
    rows=[]
    for p in FILES:
        try:
            df=load_csv(p)
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=len(df),columnas=len(df.columns),nombres_columnas=' | '.join(map(str,df.columns)),nulos=int(df.isna().sum().sum()),error=''))
        except Exception as e:
            rows.append(dict(archivo=str(p.relative_to(ROOT)),filas=None,columnas=None,nombres_columnas='',nulos=None,error=repr(e)))
    return pd.DataFrame(rows)


CSV encontrados: 86


In [6]:
import re
candidates=[p for p in FILES if 'comparacion' in p.stem.lower()]
print('CSV consolidados:',*[str(p.relative_to(ROOT)) for p in candidates],sep='\n')
if not candidates: raise ValueError('No hay tabla consolidada: se requiere identificar archivos de métricas.')
SOURCE=next((p for p in candidates if 'comparacion_deteccion' in p.stem.lower()),candidates[0]);df=load_csv(SOURCE)
display(df.head());print(df.columns.tolist())


CSV consolidados:
metricas\comparacion_deteccion.csv


,metodo,config,archivo,n,n_alerta,fraccion_alerta,episodios_visibles,episodios_detectados,alertas_fuera_etiqueta
0,DT,FULL,calibration,18691,94,0.005029,NaN,NaN,NaN
1,DT,FULL,test2,82770,998,0.012058,17.0,10.0,833.0
2,DT,FULL,test3,62370,1048,0.016803,10.0,7.0,440.0
3,DT,FULL,test4,129570,23217,0.179185,24.0,24.0,22190.0
4,DT,FULL,train2,201570,196305,0.973880,NaN,NaN,NaN


['metodo', 'config', 'archivo', 'n', 'n_alerta', 'fraccion_alerta', 'episodios_visibles', 'episodios_detectados', 'alertas_fuera_etiqueta']


In [7]:
# Identificar columnas antes de calcular: el notebook no adivina dimensiones ni métricas.
MODEL_COL = None  # nombre exacto de columna del algoritmo, o None si la configuración está unificada
ABLATION_COL = None  # nombre exacto de columna FULL/SIN_P1...SIN_P4
TEST_COL = None  # nombre exacto de TEST, si existe
METRIC_COLS = []  # métricas numéricas comparables, validadas manualmente
print('Columnas disponibles:',list(df.columns))


Columnas disponibles: ['metodo', 'config', 'archivo', 'n', 'n_alerta', 'fraccion_alerta', 'episodios_visibles', 'episodios_detectados', 'alertas_fuera_etiqueta']


In [8]:
if not ABLATION_COL or not METRIC_COLS:
 print('PENDIENTE: configurar ABLATION_COL y METRIC_COLS tras auditar la estructura real. No se calculan diferencias sin correspondencia verificable.')
else:
 keys=[c for c in (MODEL_COL,TEST_COL) if c]
 assert ABLATION_COL in df.columns and all(c in df.columns for c in keys+METRIC_COLS)
 assert all(pd.api.types.is_numeric_dtype(df[c]) for c in METRIC_COLS)
 assert not df.duplicated(keys+[ABLATION_COL]).any(),'Duplicados por modelo/TEST/ablación: resolver antes de comparar'
 base=df.loc[df[ABLATION_COL].astype(str).str.upper().eq('FULL'),keys+METRIC_COLS].set_index(keys) if keys else df.loc[df[ABLATION_COL].astype(str).str.upper().eq('FULL'),METRIC_COLS]
 assert len(base)>0,'No hay FULL'; assert keys,'Es necesario identificar columna de modelo y/o TEST para emparejar.'
 result=[]
 for abl in ('SIN_P1','SIN_P2','SIN_P3','SIN_P4'):
    sub=df.loc[df[ABLATION_COL].astype(str).str.upper().eq(abl),keys+METRIC_COLS].set_index(keys)
    aligned=base.join(sub,how='inner',lsuffix='_FULL',rsuffix='_ABL')
    for idx,r in aligned.iterrows():
     for m in METRIC_COLS:
      b=r[m+'_FULL'];a=r[m+'_ABL']
      result.append({'clave':str(idx),'ablacion':abl,'metrica':m,'FULL':b,'SIN':a,'delta_SIN_menos_FULL':a-b,'delta_relativo_pct':(a-b)/abs(b)*100 if pd.notna(b) and b!=0 else np.nan})
 result=pd.DataFrame(result);result.to_csv(DEST/'07_diferencias_ablaciones.csv',index=False,encoding='utf-8-sig');display(result)
 if len(result):
  for metric,grp in result.groupby('metrica'):
   fig,ax=plt.subplots(figsize=(8,4));grp.boxplot(column='delta_SIN_menos_FULL',by='ablacion',ax=ax)
   plt.suptitle('');ax.set_title(str(metric));fig.tight_layout();fig.savefig(DEST/f'07_ablacion_{METRIC_COLS.index(metric):02d}.png',dpi=140);plt.close(fig)
 print('Interpretar dirección de la métrica y consistencia entre TEST antes de contrastar la hipótesis.')


PENDIENTE: configurar ABLATION_COL y METRIC_COLS tras auditar la estructura real. No se calculan diferencias sin correspondencia verificable.
